# News data workflow

In [16]:
# Import required libraries
import newsapi
import dlt
import duckdb
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv
import time
from newspaper import Article
from dateutil import parser

def to_naive_iso(dt):
    if dt is None:
        return None
    # parse strings or accept datetimes
    dt = parser.isoparse(dt) if isinstance(dt, str) else dt
    if getattr(dt, "tzinfo", None):
        dt = dt.astimezone(tz=None)  # convert to local tz if desired
        dt = dt.replace(tzinfo=None)
    return dt.isoformat()

# Load environment variables
load_dotenv()

print("✅ All libraries imported successfully!")
print(f"Current working directory: {os.getcwd()}")


✅ All libraries imported successfully!
Current working directory: /Users/marcogaudio/github/data-engineering-environment-news/notebooks


In [17]:
pipeline = dlt.pipeline(
    # how the pipeline will be named in the DLT UI
    pipeline_name="news_full_content_data",
    # where the data will be stored, in this case we are using DuckDB
    destination="duckdb",
    # the name of the schema where the data will be stored in DuckDB
    dataset_name="news_data"
)

print("✅ DLT pipeline initialized successfully!")
print(f"Pipeline name: {pipeline.pipeline_name}")
print(f"Destination: {pipeline.destination}")
print(f"Dataset: {pipeline.dataset_name}")

✅ DLT pipeline initialized successfully!
Pipeline name: news_full_content_data
Destination: <dlt.destinations.duckdb(destination_type='duckdb', staging_dataset_name_layout='%s_staging', enable_dataset_name_normalization=True, info_tables_query_threshold=1000, truncate_tables_on_staging_destination_before_load=True, local_dir='/Users/marcogaudio/github/data-engineering-environment-news/notebooks', pipeline_name='news_full_content_data', pipeline_working_dir='/Users/marcogaudio/.dlt/pipelines/news_full_content_data', create_indexes=False)>
Dataset: news_data


In [ ]:
@dlt.resource(write_disposition="append")
def news_with_full_content(days_back: int = 15):    
    # loading api key from .env
    api_key = os.getenv("NEWS_API_KEY")
    if not api_key:
        # handling missing API key scenario
        raise ValueError("NEWS_API_KEY not found in environment variables.")
    
    api_client = newsapi.NewsApiClient(api_key=api_key)

    # testing API connection 
    try:
        sources = api_client.get_sources()
        print(f"✅ Connesso! Fonti disponibili: {len(sources['sources'])}")
    except Exception as e:
        print(f"❌ Errore durante la connessione all'API: {e}")

    today = datetime.now()
    limit = today - timedelta(days=days_back)
    # Fetching news data  
    try:
        
        all_articles = api_client.get_everything(
            q="Artificial Intelligence",
            from_param=limit.strftime('%Y-%m-%d'),
            to=today.strftime("%Y-%m-%d"),
            language="en",
            sort_by="relevancy",
            page_size=100
        )

        print(f"✅ Dati recuperati! Numero di articoli: {len(all_articles['articles'])}")
    
    except Exception as e:
        print(f"❌ Errore durante il recupero dei dati: {e}")

    for art in all_articles['articles']:
        try:
            # Scrape full content
            article = Article(art['url'])
            article.download()
            article.parse()

            if len(article.text.split()) < 200:  # Skip articles with very short content
                print(f"⚠️ Skip {art['url']}: content too short")
                continue
            
            # add transformation and casting here ...
            
            # remove extra spaces and newlines from full_text
            article.text = article.text.strip()
            # cast publishedAt to naive ISO format
            art['publishedAt'] = to_naive_iso(art['publishedAt'])

            yield {
                'title': art['title'],
                'source': art['source']['name'],
                'author': art['author'],
                'published_at': art['publishedAt'],
                'url': art['url'],
                'full_text': article.text,
                'word_count': len(article.text.split()),
                'top_image': article.top_image,
                'extracted_at': datetime.now().isoformat()
            }
            
            time.sleep(1)
            
        except Exception as e:
            print(f"⚠️ Skip {art['url']}: {e}")
            continue

In [19]:
# Carica
load_info = pipeline.run(
    news_with_full_content(),
    table_name="news_table",
    write_disposition="replace"
)

print(f"loaded info: {load_info}")
# Get the database path and connect directly to DuckDB
db_path = pipeline.sql_client().credentials.database
print(f"\n🔗 Database path: {db_path}")

✅ Connesso! Fonti disponibili: 125
✅ Dati recuperati! Numero di articoli: 97
⚠️ Skip https://www.theverge.com/news/936945/pope-leo-letter-encyclical-ai-anthropic-labor-warfare: content too short
⚠️ Skip https://gizmodo.com/mistral-ceo-says-the-popes-comments-are-a-big-problem-for-europes-war-on-american-tech-2000764700: Article `download()` failed with Website protected with Cloudflare, url: None on URL https://gizmodo.com/mistral-ceo-says-the-popes-comments-are-a-big-problem-for-europes-war-on-american-tech-2000764700
⚠️ Skip https://www.politico.com/news/2026/05/22/trump-concerns-ai-policy-00934622: Article `download()` failed with Website protected with Cloudflare, url: None on URL https://www.politico.com/news/2026/05/22/trump-concerns-ai-policy-00934622
⚠️ Skip https://gizmodo.com/99-of-ceos-expect-ai-driven-layoffs-in-the-next-two-years-2000762994: Article `download()` failed with Website protected with Cloudflare, url: None on URL https://gizmodo.com/99-of-ceos-expect-ai-driven-

In [23]:
# Query DuckDB
con = duckdb.connect(db_path)

# Get table name with proper schema handling
all_tables = con.execute("""
    SELECT table_schema, table_name 
    FROM information_schema.tables 
    WHERE table_type = 'BASE TABLE' AND table_schema != 'information_schema'
""").fetchdf()

print(f"\n📋 Available tables:")
print(all_tables)


📋 Available tables:
  table_schema           table_name
0    news_data           news_table
1    news_data           _dlt_loads
2    news_data  _dlt_pipeline_state
3    news_data         _dlt_version


In [24]:
# removing dlt default tables
data_tables = all_tables[~all_tables['table_name'].str.startswith('_dlt')]

# if no tables found, print message and exit
if data_tables.empty:
    print("❌ No data tables found. Please run the data loading cell first.")
else:
    # Assuming we want to query the first available data table
    table_schema = data_tables.iloc[0]['table_schema']
    table_name = data_tables.iloc[0]['table_name']
    full_table_name = f"{table_schema}.{table_name}"
    
    print(f"\n📊 Using table: {full_table_name}")

# Query the data
# first query: basic summary statistics

print("\n📈 Summary Statistics:")
summary_stats_query = (f"""
    SELECT 
        COUNT(*) AS total_articles,
        AVG(word_count) AS avg_word_count,
        MAX(word_count) AS max_word_count,
        MIN(word_count) AS min_word_count,
        MIN(published_at) AS earliest_article,
        MAX(published_at) AS latest_article,
        COUNT(DISTINCT source) AS unique_sources,
        COUNT(DISTINCT author) AS unique_authors
    FROM {full_table_name}
""")

summary_result = con.execute(summary_stats_query).fetchdf()
print(summary_result.to_string(index=False))

con.close()
print("\n✅ DuckDB connection closed")


📊 Using table: news_data.news_table

📈 Summary Statistics:
 total_articles  avg_word_count  max_word_count  min_word_count          earliest_article            latest_article  unique_sources  unique_authors
             66       722.69697            1928             226 2026-05-16 17:12:57+02:00 2026-05-30 21:02:54+02:00              28              56

✅ DuckDB connection closed


In [25]:
# writing results to Excel
# creating output directory if it doesn't exist
os.makedirs("../output", exist_ok=True)

print("\n💾 Saving summary statistics to CSV...")
summary_result.to_csv("../output/summary_statistics.csv", index=False)
print("✅ Summary statistics saved to CSV.")


💾 Saving summary statistics to CSV...
✅ Summary statistics saved to CSV.
